<img src="https://github.com/IKNL/guidelines/blob/master/resources/logos/iknl_nl.png?raw=true" width=200 align="right">

# Crosstabs
**Phase 1: Fake OMOP data in UPM and IKNL**

In [1]:
# The only analytics that we are going to show in this demo is the crosstabs.
#
# AUTHENTICATION
# --------------------------------------------------------------------------------------
# In this notebook we first authenticate to obtain a JWT token which can be used for the
# successive calls. This authentication is *not* coupled to the CERTH Keycloak. The
# token returned is set to expire after 5 days for this demo so we do not need to
# re-authenticate to often. I suggest to hard-code the token in the RAVEN UI/API for
# now.
#
# CROSSTABS ALGORITHM
# --------------------------------------------------------------------------------------
# The crosstabs algorithm computes the contingency table between two or more categorical
# variables. I suggest to have a brief look at the swimlane diagram:
# https://algorithms.vantage6.ai/en/latest/v6-crosstab-py/docs/v6-crosstab-py/implementation.html#overview
# to have a good overview of the different steps in the algorithm. From the diagram
# you can see that this is a one (federated-)step algorithm:
#
# 1. Call `partial_crosstab`
#
# And then there is the central part responsible for the aggregation of the results. The
# central part of the algorithm (the main call) will return the crosstabs for the entire
# federated dataset. In IDEA4RC, the crosstabs per data station are also required. So in
# this notebook we go through the following steps to obtain both the *global* (from
# the central part) and the *local* (from the `partial_crosstab` call) crosstabs:
#
# 1. Create a new vantage6 task to execute the *crosstabs* method (central part). This
#    central part will start the task `partial_crosstab` (as you can see in the
#    swimlane diagram).
# 2. Poll until the task is finished
# 3. Retrieve the *global* crosstabs from the central part (the main call)
# 4. Retrieve the *local* crosstabs from the data stations (the `partial_crosstab`
#    call that was made by the central part)
#

In [2]:
import requests
import json
import base64

from vantage6.client import UserClient

## Authentication

In [3]:
client = UserClient(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es:443/server",
    auth_url="https://vantage6-auth.orchestrator.idea.lst.tfo.upm.es:443",
    auth_client="public_client",
    auth_realm="vantage6",
    log_level="INFO"
)
# You can authenticate using the user `itziar` and the password that I've send to you.
client.authenticate()

# Set the headers for the other requests
headers = {
    "Authorization": f"Bearer {client._access_token}"
}

# Print the server version
print("Server version: ", client.util.get_server_version())

 Welcome to
                  _                     __  
                 | |                   / /  
__   ____ _ _ __ | |_ __ _  __ _  ___ / /_  
\ \ / / _` | '_ \| __/ _` |/ _` |/ _ \ '_ \ 
 \ V / (_| | | | | || (_| | (_| |  __/ (_) |
  \_/ \__,_|_| |_|\__\__,_|\__, |\___|\___/ 
                            __/ |           
                           |___/            

 --> Join us on Discord! https://discord.gg/rwRvwyK
 --> Docs: https://docs.vantage6.ai
 --> Blog: https://vantage6.ai
------------------------------------------------------------
Cite us!
If you publish your findings obtained using vantage6, 
please cite the proper sources as mentioned in:
https://vantage6.ai/vantage6/references
------------------------------------------------------------
Opening browser for login


127.0.0.1 - - [18/Nov/2025 11:42:45] "GET /callback?state=state&session_state=fcab9cc2-e144-4b80-9d57-c954a3d9158f&iss=https%3A%2F%2Fvantage6-auth.orchestrator.idea.lst.tfo.upm.es%2Frealms%2Fvantage6&code=61515e2a-ac67-4ec8-83b7-7eae21f4e77a.fcab9cc2-e144-4b80-9d57-c954a3d9158f.fcb015fe-d5b0-4a7b-b609-7d87a3b72f3e HTTP/1.1" 200 -


 --> Succesfully authenticated
 --> Name: admin (id=1)
 --> Organization: root (id=1)
Server version:  {'version': '5.0.0a43'}


In [ ]:
headers

## Crosstabs Algorithm

In [5]:
# The vantage6 server requires a certain payload to the request. It requires:
#
# * The STUDY_ID (which is implicit also defining the COLLABORATION_ID)
# * The SESSION_ID
# * The ORG_IDS (all the organizations that should be included in the analysis)
# * IMAGE (the docker image to use that contains the summary algorithm)
# * METHOD (the method to execute)
#
# For this demo (in phase 1) we hard-code all of these values.
#
#
# This study is part of collaboration 2, and consists of UPM and IKNL
STUDY_ID = 3
#
# A session that is already part of the study is 2
SESSION_ID = 2
#
# The image to use is the latest version of the sessions algorithm
IMAGE = "harbor2.vantage6.ai/idea4rc/analytics:latest"
#
# The method (that is within this IMAGE) to execute is the summary algorithm
METHOD = "crosstab"
#
# Organization IDs for UPM and IKNL
UPM_ORG_ID = 3
IKNL_ORG_ID = 1
ORG_IDS = [UPM_ORG_ID, IKNL_ORG_ID]

In [6]:
# Besides that the vantage6 server expect a certain payload the algorithm also
# expects certain input:
#
# * DATAFRAME_IDS (the IDs of the dataframes to include in the analysis)
# * RESULTS_COL (the variable to compute the crosstab for)
# * GROUP_COLS (the variable to group the data by)
#
# These dataframes are already created and part of the session/study.
pelvis = 76 # Pelvis cohort ID
rps_pelvis = 77 # RPS+Pelvis cohort ID
rps = 78 # RPS cohort ID
DATAFRAME_IDS = [pelvis, rps_pelvis, rps]

# These are the variable names that we want to compute the contingency table between.
# Note that it is possible to provide a list for the GROUP_COLS, but for now lets keep
# it simple and compute the crosstab for a two variables.
RESULTS_COL = "sex"
GROUP_COLS = ["fnclcc_grade"]

org_input = [
    {
        "id": UPM_ORG_ID, # Central task is executed by UPM
        "arguments": base64.b64encode(
            json.dumps(
                {
                    "results_col": RESULTS_COL,
                    "group_cols": GROUP_COLS,
                    "organizations_to_include": ORG_IDS
                }
            ).encode("UTF-8")
        ).decode("UTF-8")
    }
]

In [7]:
payload = {
    "name": "Human-readable name of the task",
    "image": IMAGE,
    "description": "Description of the task",
    "action": "central_compute",
    "method": METHOD,
    "organizations": org_input,
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": df_id
            } for df_id in DATAFRAME_IDS
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}

In [8]:
# -------------------------------------------------------------------------------------
#|  You can simply copy the following payload into the RAVEN UI for now. In the future |
#|  some of the values that we now hard-coded should be replaced by the RAVEN UI.      |
# -------------------------------------------------------------------------------------
payload

{'name': 'Human-readable name of the task',
 'image': 'harbor2.vantage6.ai/idea4rc/analytics:latest',
 'description': 'Description of the task',
 'action': 'central_compute',
 'method': 'crosstab',
 'organizations': [{'id': 3,
   'arguments': 'eyJyZXN1bHRzX2NvbCI6ICJzZXgiLCAiZ3JvdXBfY29scyI6IFsiZm5jbGNjX2dyYWRlIl0sICJvcmdhbml6YXRpb25zX3RvX2luY2x1ZGUiOiBbMywgMV19'}],
 'databases': [[{'type': 'dataframe', 'dataframe_id': 76},
   {'type': 'dataframe', 'dataframe_id': 77},
   {'type': 'dataframe', 'dataframe_id': 78}]],
 'session_id': 2,
 'study_id': 3}

In [9]:
# Then using the authorization header and the payload we can create a vantage6 task
# using the vantage6 server API.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
# In the response we need to extract the task ID and the job ID so we can poll werther
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]
response.json()

{'job_id': 86,
 'algorithm_store': None,
 'description': 'Description of the task',
 'runs': '/server/run?task_id=238',
 'finished_at': None,
 'study': {'id': 3,
  'link': '/server/study/3',
  'methods': ['DELETE', 'GET', 'PATCH']},
 'databases': [{'label': None,
   'type': 'dataframe',
   'dataframe_id': 76,
   'dataframe_name': 'Pelvis',
   'position': 0},
  {'label': None,
   'type': 'dataframe',
   'dataframe_id': 77,
   'dataframe_name': 'Pelvis_RPS',
   'position': 0},
  {'label': None,
   'type': 'dataframe',
   'dataframe_id': 78,
   'dataframe_name': 'RPS',
   'position': 0}],
 'dataframe': None,
 'required_by': [],
 'status': 'awaiting',
 'created_at': '2025-11-18T10:27:54.853677',
 'id': 238,
 'depends_on': [],
 'init_org': {'id': 1,
  'link': '/server/organization/1',
  'methods': ['DELETE', 'GET', 'PATCH']},
 'parent': None,
 'collaboration': {'id': 2,
  'link': '/server/collaboration/2',
  'methods': ['DELETE', 'GET', 'PATCH']},
 'children': '/server/task?parent_id=238',


In [10]:
# Poll (every 5 seconds or so, dont ddos the server) until the (central) task is
# finished. We do not concern about the subtasks in this instance. We could consider
# including them in the future as well?
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
# Since it is a central task we can obtain the [0]th element of the data list as it
# always should be a single element in a list. Wait until the status returns
# "completed".
response.json()["data"][0]["status"]

'pending'

In [17]:
# We need to poll when the subtask is ready (created by the central part). Once ready
# we need to obtain the ID of the subtask so we can poll the status of each individual
# subtask.
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task?parent_id={TASK_ID}",
    headers=headers,
)
if response.json()["data"]:
    SUBTASK_ID = response.json()["data"][0]["id"]
    print(SUBTASK_ID)

239


In [19]:
# Get the status of the subtasks (poll them also every 5 seconds)
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={SUBTASK_ID}",
    headers=headers,
)
[{"organization_id": container["organization"]["id"], "status": container["status"]} for container in response.json()["data"]]


[{'organization_id': 1, 'status': 'completed'},
 {'organization_id': 3, 'status': 'completed'}]

In [20]:
# Get the results of the (central) task, thus the *global* contingency table.
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/result?task_id={TASK_ID}",
    headers=headers,
)
# Again, since it is a central task we can obtain the [0]th element of the data list
json.loads(base64.b64decode(response.json()["data"][0]["result"]).decode("UTF-8"))

{'Pelvis': {'contingency_table': [{'fnclcc_grade': 'Grade 1 tumor',
    'FEMALE': '84',
    'MALE': '82',
    'Total': '166'},
   {'fnclcc_grade': 'Grade 2 tumor',
    'FEMALE': '106',
    'MALE': '78',
    'Total': '184'},
   {'fnclcc_grade': 'Grade 3 tumor',
    'FEMALE': '114',
    'MALE': '98',
    'Total': '212'},
   {'fnclcc_grade': 'N/A', 'FEMALE': '4', 'MALE': '6', 'Total': '10'},
   {'fnclcc_grade': 'Total', 'FEMALE': '308', 'MALE': '264', 'Total': '572'}],
  'chi2': {'chi2': '2.522825698669517', 'P-value': '0.4711800884562277'}},
 'Pelvis_RPS': {'contingency_table': [{'fnclcc_grade': 'Grade 1 tumor',
    'FEMALE': '208',
    'MALE': '200',
    'Total': '408'},
   {'fnclcc_grade': 'Grade 2 tumor',
    'FEMALE': '208',
    'MALE': '172',
    'Total': '380'},
   {'fnclcc_grade': 'Grade 3 tumor',
    'FEMALE': '204',
    'MALE': '196',
    'Total': '400'},
   {'fnclcc_grade': 'N/A', 'FEMALE': '4', 'MALE': '8', 'Total': '12'},
   {'fnclcc_grade': 'Total', 'FEMALE': '624', 'MALE': 